# Phase 6: Statistical Analysis & Hypothesis Testing (Housing)

**Objective:** Move beyond visual exploration to quantify the characteristics of the housing market using advanced descriptive statistics and formal hypothesis testing.

In this notebook, we will:
1. Calculate the exact shape and dispersion of our continuous variables.
2. Mathematically test our target variable (`Price`) for normality.
3. Perform a non-parametric hypothesis test (Mann-Whitney U) to prove whether Air Conditioning actually drives a statistically significant increase in home prices.
4. Discuss Correlation vs. Causation regarding confounding variables.

In [1]:
# Automatically reload our modules if we change them
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

# Add the project root to the path
sys.path.append(str(Path.cwd().parent.parent))
from src.utils.stats import check_normality, run_mann_whitney

# Load processed housing data
data_path = Path.cwd().parent.parent / "datasets" / "processed" / "housing_cleaned.csv"
df = pd.read_csv(data_path)
print(f"Housing dataset loaded successfully. Shape: {df.shape}")

Housing dataset loaded successfully. Shape: (545, 13)


### 1. Advanced Descriptive Statistics: Dispersion and Shape
During EDA, we visually observed that `Price` and `Area` were right-skewed. Now, we will quantify this. 
- **Variance/Standard Deviation:** Measures how spread out the prices are from the average.
- **Skewness:** Measures the asymmetry of the distribution (Positive = Right-skewed).
- **Kurtosis:** Measures the "tailedness" of the distribution (High kurtosis = more extreme outliers).

In [2]:
# Calculate advanced descriptive stats for key continuous variables
stats_df = pd.DataFrame({
    'Mean': df[['price', 'area']].mean(),
    'Std Dev': df[['price', 'area']].std(),
    'Variance': df[['price', 'area']].var(),
    'Skewness': df[['price', 'area']].skew(),
    'Kurtosis': df[['price', 'area']].kurtosis()
})

# Format for readability
stats_df.style.format("{:,.2f}")

,Mean,Std Dev,Variance,Skewness,Kurtosis
price,"4,766,729.25","1,870,439.62","3,498,544,355,820.57",1.21,1.96
area,"5,150.54","2,170.14","4,709,512.06",1.32,2.75


### 2. Hypothesis Testing: The Normality Assumption
Before comparing groups, we must determine if our target variable (`price`) follows a normal distribution. This acts as our "gatekeeper" to decide between parametric (T-Test) and non-parametric (Mann-Whitney U) testing.

**Hypotheses for Shapiro-Wilk:**
- $H_0$: The data is normally distributed.
- $H_1$: The data is NOT normally distributed.
- **Significance Level ($\alpha$):** 0.05

In [6]:
# Run Shapiro-Wilk Normality Test
# Check Normality for both Price and Area
price_norm = check_normality(df['price'])
area_norm = check_normality(df['area'])

print(f"Test: {price_norm['test']}")
print(f"Statistic: {price_norm['statistic']:.4f}")
print(price_norm['conclusion'])

print(f"\nTest: {area_norm['test']}")
print(f"Statistic: {area_norm['statistic']:.4f}")
print(area_norm['conclusion'])

Test: Shapiro-Wilk Normality Test
Statistic: 0.9216
Reject Null Hypothesis (p=0.0000 < 0.05): Deviation from normality is statistically significant.

Test: Shapiro-Wilk Normality Test
Statistic: 0.9113
Reject Null Hypothesis (p=0.0000 < 0.05): Deviation from normality is statistically significant.


### 3. Hypothesis Testing: Air Conditioning Impact (Mann-Whitney U)
Because our normality test failed (the data is heavily skewed), we cannot use a standard T-Test. We will use the non-parametric **Mann-Whitney U test** to compare the median ranks of homes with and without Air Conditioning.

**Hypotheses:**
- $H_0$: There is no difference in the price distributions between homes with and without AC.
- $H_1$: There is a statistically significant difference in price distributions.
- **Significance Level ($\alpha$):** 0.05

In [7]:
# Systematically test all binary categorical variables against Price using Mann-Whitney U
binary_amenities = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

print("--- SYSTEMATIC MANN-WHITNEY U TESTS ---")
for amenity in binary_amenities:
    group_yes = df[df[amenity] == 'yes']['price']
    group_no = df[df[amenity] == 'no']['price']
    
    # Run our utility function
    mw_result = run_mann_whitney(group_yes, group_no)
    
    print(f"\nFeature: {amenity.upper()}")
    print(mw_result['conclusion'])

--- SYSTEMATIC MANN-WHITNEY U TESTS ---

Feature: MAINROAD
Reject Null Hypothesis (p=0.0000 < 0.05): Difference in distributions between the two groups is statistically significant.

Feature: GUESTROOM
Reject Null Hypothesis (p=0.0000 < 0.05): Difference in distributions between the two groups is statistically significant.

Feature: BASEMENT
Reject Null Hypothesis (p=0.0000 < 0.05): Difference in distributions between the two groups is statistically significant.

Feature: HOTWATERHEATING
Reject Null Hypothesis (p=0.0461 < 0.05): Difference in distributions between the two groups is statistically significant.

Feature: AIRCONDITIONING
Reject Null Hypothesis (p=0.0000 < 0.05): Difference in distributions between the two groups is statistically significant.

Feature: PREFAREA
Reject Null Hypothesis (p=0.0000 < 0.05): Difference in distributions between the two groups is statistically significant.


### 4. Analytical Narrative: Correlation vs. Causation

During our Exploratory Data Analysis, we noted a strong positive linear correlation between `Bathrooms` and `Price`. A naive statistical interpretation would be: *"Adding a bathroom directly causes a proportional increase in home value."*

**The Confounding Variable Problem:**
While it is true that bathrooms add intrinsic value to a property, we must account for **confounding variables**. A confounding variable is an outside influence that changes the effect of a dependent and independent variable.

In this dataset, `Area` is a massive confounding variable. Larger homes naturally have the physical footprint required to support 3 or 4 bathrooms. Therefore, `Bathrooms` and `Area` are highly collinear (they move together). 

If we tell a business stakeholder that "adding a bathroom increases price by X," we are likely overestimating the bathroom's value, because that historical price increase was driven simultaneously by the fact that the house was simply larger. In machine learning, this collinearity can cause algorithms like Multiple Linear Regression to assign unstable, inaccurate weights to our features.

### 5. Comprehensive Phase 6 Conclusion (Housing)

By applying exhaustive inferential statistics, we have exposed critical structural realities of the housing dataset that visual EDA alone could not definitively prove.

**1. The Failure of Central Tendency & Normality:**
Our descriptive statistics and the Shapiro-Wilk test officially rejected the normality assumption ($p < 0.05$) for both `Price` and `Area`. Both variables possess severe positive skewness (>1.0) and high kurtosis. Consequently, the mean is a highly inaccurate measure of central tendency for this market. This mathematical proof legally bars us from using standard parametric tests (like the Student's T-Test). 

**2. Systematic Validation of Amenities (The Weak Signal):**
By programmatically running the non-parametric Mann-Whitney U test across all binary categorical features, we mathematically proved that every tested amenity creates a statistically significant difference in median home prices. However, `hotwaterheating` barely passed the significance threshold ($p=0.0461 < 0.05$). While statistically significant, it represents our weakest signal and may be a candidate for pruning to prevent model complexity.

**3. The Confounding Variable Problem:**
While features like `Bathrooms` and `guestroom` correlate with higher prices, we must acknowledge `Area` as a massive confounding variable. Larger homes naturally support more amenities. This collinearity means we cannot naively assume that adding an amenity in isolation will guarantee a proportional price spike.

**4. Architectural Implications for Modeling:**
Because of the massive variance and extreme right-skewed outliers, standard data scaling techniques like `MinMaxScaler` or `StandardScaler` will likely fail—they will crush 95% of our normal homes into a tiny decimal range. To build a robust predictive model, we must use outlier-resistant scaling techniques (like `RobustScaler`) and consider log-transforming our `Price` and `Area` columns.